In [ ]:
import fitz  # PyMuPDF
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pytesseract
import easyocr


# Torniamo a PyMuPDF

In [ ]:
# Convertire le immagini in testo

doc = fitz.open("../Lezione2/in/Ricevute.pdf")
print(f"Ricevute.png: Pagine totali: {len(doc)}")
print(f"Metadati: {doc.metadata}")

pagina = doc[0] # Prendi la prima pagina

rect=fitz.Rect(15,20,160,50)
pagina.set_cropbox(rect=rect)

for zoom in [2, 4, 8, 16]:
    testo = pagina.get_textpage_ocr(dpi=72*zoom)
    print(f'DPI: {zoom*72}')
    print(testo.extractTEXT())

# Lettura di modello prestampato suggerito da Gemini

In [ ]:
import ipywidgets as widgets
from IPython.display import display

from snippets.modules.pipeline import Pipeline
import PIL.Image


# Caricamento del modello vuoto
img_empty = cv2.imread("../Lezione2/in/modello.jpg")

# --- 1. RENDERIZZAZIONE DELLA PAGINA PDF AD ALTA RISOLUZIONE ---
doc = fitz.open("../Lezione2/in/modello-riempito-300dpi.pdf")

zoom = 4.16
mat = fitz.Matrix(zoom, zoom)


# Inizializziamo il lettore EasyOCR impostando la lingua italiana.
# Nota: gpu=True accelera drasticamente l'elaborazione se hai una scheda video Nvidia supportata.
easyocr_reader = easyocr.Reader(['it'], gpu=False)

def appy_to_page(page):
    # Usiamo una matrice di zoom per raggiungere circa 300 DPI (72 DPI * 4.16 ≈ 300 DPI)
    pix = page.get_pixmap(matrix=mat)

    # Conversione in array NumPy (BGR) per OpenCV
    img_filled_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape((pix.h, pix.w, pix.n))
    img_filled = cv2.cvtColor(img_filled_data, cv2.COLOR_RGBA2BGR if pix.n == 4 else cv2.COLOR_RGB2BGR)

    img_empty_resized = cv2.resize(img_empty, (img_filled.shape[1], img_filled.shape[0]))

    # Convertiamo entrambe le immagini in scala di grigi
    gray_filled = cv2.cvtColor(img_filled, cv2.COLOR_BGR2GRAY)
    gray_empty = cv2.cvtColor(img_empty_resized, cv2.COLOR_BGR2GRAY)

    gray_filled_pil = Pipeline(gray_filled[200:600, 150:1000]).run(to="PIL")
    gray_empty_pil = Pipeline(gray_empty[200:600, 150:1000]).run(to="PIL")
    image_original.update(gray_filled_pil)
    image_model.update(gray_empty_pil)

    # --- 3. ALLINEAMENTO DEL MODELLO VUOTO SUL DOCUMENTO REALE (Registration) ---
    # Inizializziamo il rilevatore di punti chiave ORB
    orb = cv2.ORB_create(nfeatures=5000)
    kp_empty, des_empty = orb.detectAndCompute(gray_empty, None)
    kp_filled, des_filled = orb.detectAndCompute(gray_filled, None)

    # Matcher basato su Brute-Force Hamming (ideale per ORB)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des_empty, des_filled)
    matches = sorted(matches, key=lambda x: x.distance)

    # Estraiamo le coordinate dei migliori match (top 20%)
    good_matches = matches[:int(len(matches) * 0.2)]
    src_pts = np.float32([kp_empty[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp_filled[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)


    # Calcoliamo la matrice di Omografia per distorcere e adattare il modello vuoto
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    # Allineiamo geometricamente il modello vuoto alle dimensioni esatte del foglio compilato
    height, width = gray_filled.shape
    aligned_empty = cv2.warpPerspective(gray_empty, H, (width, height))

    # --- 4. SOTTRAZIONE DEL MODELLO PER ISOLARE IL TESTO SCRITTO ---
    # Applichiamo una binarizzazione a entrambe le immagini prima del confronto
    _, bin_filled = cv2.threshold(gray_filled, 200, 255, cv2.THRESH_BINARY_INV)
    _, bin_empty = cv2.threshold(aligned_empty, 200, 255, cv2.THRESH_BINARY_INV)

    # Dilatiamo leggermente la maschera del modello vuoto. 
    # Questo serve a tollerare micro-errori di allineamento evitando bave nere attorno alle scritte fisse
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    bin_empty_dilated = cv2.dilate(bin_empty, kernel, iterations=1)

    # SOTTRAZIONE: Rimuoviamo tutto ciò che appartiene al modello vuoto
    handwritten_only = cv2.subtract(bin_filled, bin_empty_dilated)


    handwritten_only_filtered = Pipeline(handwritten_only) \
        .gaussian_blur(ksize=gaussian_ksize_value, sigma_x=gaussian_sigma_x_value, sigma_y=gaussian_sigma_y_value) \
        .run(to=np.ndarray)

    # Invertiamo il risultato per avere testo nero su sfondo bianco (ottimale per Tesseract)
    ocr_ready = cv2.bitwise_not(handwritten_only_filtered)

    # Pulizia finale da piccoli pixel isolati (rumore di fondo)
    ocr_ready = cv2.medianBlur(ocr_ready, 1)

    handwritten_pil = Pipeline(ocr_ready[200:600, 150:1000]).run(to="PIL")
    image_handwrite.update(handwritten_pil)

    if ocr_active:
        # --- 5. RICONOSCIMENTO OCR DEL SOLO TESTO COMPILATO ---
        tesseract_config = r'--oem 3 --psm 11 -l ita'
        testo_estratto = pytesseract.image_to_string(ocr_ready, config=tesseract_config)


        print(f"--- TESTO COMPILATO PAGINA ESTRATTO (SENZA STRUTTURA) ---")
        print(testo_estratto)

        testo_e_dati = pytesseract.image_to_data(ocr_ready, config=tesseract_config, output_type=pytesseract.Output.DICT)
        for i in range(len(testo_e_dati['text'])):
            if int(testo_e_dati['conf'][i]) > 60:  # Filtro per confidenza
                testo = testo_e_dati['text'][i]
                confidenza = testo_e_dati['conf'][i]
                left = testo_e_dati['left'][i]
                top = testo_e_dati['top'][i]

                print(f"Testo: {testo}, Confidenza: {confidenza} a pos: {left},{top}")




        # Eseguiamo la lettura direttamente sull'array OpenCV (ocr_ready)
        # paragraph=True unisce le parole vicine sulla stessa riga logica
        risultati = easyocr_reader.readtext(ocr_ready)

        print("--- DATI COMPILATI ESTRATTI CON EASYOCR ---")
        for res in risultati:
            coordinate_box = res[0]  # Gli angoli del rettangolo di testo trovato
            testo = res[1]           # Il testo riconosciuto
            
            print(f"{coordinate_box} - {testo}")

ocr_active = False

page_input = widgets.IntSlider(
    value=1,
    min=1,
    max=len(doc),
    step=1,
    description='Pagina:')

ocr_input = widgets.Checkbox(
    value=ocr_active,
    description='Esegui OCR',
    disabled=False)

filters = Pipeline().available_filters()

gaussian_ksize_value = filters['gaussian_blur']['params']['ksize']
gaussian_sigma_x_value = filters['gaussian_blur']['params']['sigma_x']
gaussian_sigma_y_value = filters['gaussian_blur']['params']['sigma_y']

gaussian_ksize = widgets.IntSlider(
    value=filters['gaussian_blur']['params']['ksize'],
    min=filters['gaussian_blur']['ranges']['ksize'][0],
    max=filters['gaussian_blur']['ranges']['ksize'][1],
    step=2,
    description='Gaussian ksize:'
)

gaussian_sigma_x = widgets.FloatSlider(
    value=filters['gaussian_blur']['params']['sigma_x'],
    min=filters['gaussian_blur']['ranges']['sigma_x'][0],
    max=filters['gaussian_blur']['ranges']['sigma_x'][1],
    step=0.1,
    description='Gaussian sigma_x:'
)

gaussian_sigma_y = widgets.FloatSlider(
    value=filters['gaussian_blur']['params']['sigma_y'],
    min=filters['gaussian_blur']['ranges']['sigma_y'][0],
    max=filters['gaussian_blur']['ranges']['sigma_y'][1],
    step=0.1,
    description='Gaussian sigma_y:'
)



def update_page_number(new_page):
    page = doc[new_page - 1]  # Le pagine sono indicizzate da 0
    appy_to_page(page)

def update_ocr_status(ocr_enabled):
    global ocr_active
    ocr_active = ocr_enabled

def update_gaussian_ksize_value(new_ksize):
    global gaussian_ksize_value
    gaussian_ksize_value = new_ksize
    appy_to_page(page)

def update_gaussian_sigma_x_value(new_sigma_x):
    global gaussian_sigma_x_value
    gaussian_sigma_x_value = new_sigma_x
    appy_to_page(page)

def update_gaussian_sigma_y_value(new_sigma_y):
    global gaussian_sigma_y_value
    gaussian_sigma_y_value = new_sigma_y
    appy_to_page(page)

page_input.observe(lambda change: update_page_number(change['new']), names='value')
ocr_input.observe(lambda change: update_ocr_status(change['new']), names='value')
gaussian_ksize.observe(lambda change: update_gaussian_ksize_value(change['new']), names='value')
gaussian_sigma_x.observe(lambda change: update_gaussian_sigma_x_value(change['new']), names='value')
gaussian_sigma_y.observe(lambda change: update_gaussian_sigma_y_value(change['new']), names='value')

display(ocr_input)
display(page_input)
display(gaussian_ksize)
display(gaussian_sigma_x)
display(gaussian_sigma_y)


page = doc[0]  # Prendiamo la prima pagina del PDF

image_handwrite = display(None, display_id=True)
image_original = display(None, display_id=True)
image_model = display(None, display_id=True)
text_extracted = display(None, display_id=True)

appy_to_page(page)


In [ ]:
display(Pipeline().available_filters())